# 07 — Temporal Coupling

This notebook implements T3 / P5: persistent super-node identity across 2020 → 2022 → 2024 → 2026 and a temporal event log for birth / growth / merge / split / death. The task explicitly allows matching as the temporal-coupling mechanism, but requires us to state what it enforces and what it costs.

Our mechanism will be:

static hierarchy at $$ t→overlap\ +\ parent-aware\ matching→persistent\ identitie $$

Importantly, new nodes are excluded from the overlap denominator when matching identities, so a cluster does not lose its identity merely because new literature was added.

## Clone repository

In [1]:
from pathlib import Path

REPO_DIR = Path("/content/tkh-hierarchy-project")

if not REPO_DIR.exists():
    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git
else:
    print("Repository already cloned.")

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 57 (delta 18), reused 44 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 16.54 MiB | 42.23 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [2]:
%cd /content/tkh-hierarchy-project

/content/tkh-hierarchy-project


## Imports

In [3]:
import json
import time

from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment

## Paths and configuration

In [4]:
PROJECT_DIR = Path(
    "/content/tkh-hierarchy-project"
)

STATIC_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "hierarchy"
    / "static"
)

TEMPORAL_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "hierarchy"
    / "temporal"
)

TEMPORAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]


LEVELS = [
    0,
    1,
    2
]


# Minimum raw membership similarity required
# for identity inheritance.
MATCH_THRESHOLD = 0.20


# Threshold used to detect split/merge relations.
EVENT_OVERLAP_THRESHOLD = 0.20


# Parent consistency is used as a small
# top-down identity-matching preference.
PARENT_BONUS = 0.15


print(
    "Static hierarchy directory:",
    STATIC_DIR
)

print(
    "Temporal output directory:",
    TEMPORAL_DIR
)

Static hierarchy directory: /content/tkh-hierarchy-project/artifacts/hierarchy/static
Temporal output directory: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal


## Method explanation

## Temporal-coupling mechanism

The static hierarchies from notebook 06 were constructed independently for each
snapshot. This notebook couples those hierarchies through persistent super-node
identities.

For consecutive snapshots $t$ and $t+1$, cluster identity is matched using only
nodes already visible at time $t$.

Let $C_i^t$ be an old cluster and $C_j^{t+1}$ a new cluster. Define the common
node universe as

$$
V_{\mathrm{common}}
=
V_t \cap V_{t+1}.
$$

Because the TKH grows monotonically under our snapshot definition,

$$
V_t \subseteq V_{t+1},
$$

so in practice $V_{\mathrm{common}}=V_t$.

The new cluster is restricted to previously visible nodes before similarity is
computed:

$$
\widetilde C_j^{t+1}
=
C_j^{t+1}
\cap
V_t.
$$

Identity similarity is then measured using Jaccard overlap:

$$
J_{ij}
=
\frac{
|C_i^t \cap \widetilde C_j^{t+1}|
}{
|C_i^t \cup \widetilde C_j^{t+1}|
}.
$$

This prevents newly introduced corpus nodes from artificially lowering temporal
identity similarity.

## Parent-aware matching objective

### Parent-aware identity matching

The hierarchy is matched from coarse to fine:

$$
P_0 \rightarrow P_1 \rightarrow P_2.
$$

At level 0, the matching score is simply

$$
M_{ij}=J_{ij}.
$$

For levels 1 and 2, we add a small bonus when the old and new clusters belong to
the same already-matched persistent parent:

$$
M_{ij}
=
(1-\eta)J_{ij}
+
\eta
\mathbf{1}
\left[
\operatorname{parent}(C_i^t)
=
\operatorname{parent}(C_j^{t+1})
\right],
$$

where

$$
\eta=0.15.
$$

The Hungarian algorithm maximizes the total matching score.

Importantly, persistent identity is inherited only when the **raw membership
Jaccard score** exceeds the acceptance threshold. Parent agreement alone can
therefore never create a false temporal identity.

## Load the static hierarchies

In [5]:
def load_static_hierarchy(year):

    path = (
        STATIC_DIR
        /
        f"hierarchy_{year}_static_unlabelled.json"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing static hierarchy: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)

In [6]:
static_hierarchies = {
    year: load_static_hierarchy(year)
    for year in SNAPSHOT_YEARS
}


for year in SNAPSHOT_YEARS:

    hierarchy = static_hierarchies[
        year
    ]

    print(
        year,
        {
            level:
                len(
                    hierarchy[
                        "levels"
                    ][str(level)]
                )

            for level in LEVELS
        }
    )

2020 {0: 12, 1: 60, 2: 300}
2022 {0: 12, 1: 60, 2: 300}
2024 {0: 12, 1: 60, 2: 300}
2026 {0: 12, 1: 60, 2: 300}


## Basic static hierarchy validation

In [7]:
def records_by_id(
    hierarchy,
    level
):
    records = (
        hierarchy[
            "levels"
        ][str(level)]
    )

    result = {
        record["id"]: record
        for record in records
    }

    assert (
        len(result)
        ==
        len(records)
    )

    return result

In [8]:
for year in SNAPSHOT_YEARS:

    for level in LEVELS:

        records = records_by_id(
            static_hierarchies[year],
            level
        )

        print(
            year,
            f"P{level}",
            len(records)
        )

2020 P0 12
2020 P1 60
2020 P2 300
2022 P0 12
2022 P1 60
2022 P2 300
2024 P0 12
2024 P1 60
2024 P2 300
2026 P0 12
2026 P1 60
2026 P2 300


## Construct node → cluster assignments

In [9]:
def node_assignment(
    hierarchy,
    level
):
    assignment = {}


    for record in (
        hierarchy[
            "levels"
        ][str(level)]
    ):

        cluster_id = (
            record["id"]
        )


        for node_id in (
            record["member_ids"]
        ):

            if node_id in assignment:

                raise ValueError(
                    f"Node {node_id} appears "
                    f"twice at level {level}."
                )


            assignment[
                node_id
            ] = cluster_id


    return assignment

## Validate monotonic node availability

In [10]:
for old_year, new_year in zip(
    SNAPSHOT_YEARS[:-1],
    SNAPSHOT_YEARS[1:]
):

    old_nodes = set(
        node_assignment(
            static_hierarchies[
                old_year
            ],
            2
        ).keys()
    )

    new_nodes = set(
        node_assignment(
            static_hierarchies[
                new_year
            ],
            2
        ).keys()
    )


    assert (
        old_nodes
        <=
        new_nodes
    )


    print(
        f"{old_year} -> {new_year}:",
        f"+{len(new_nodes - old_nodes):,} nodes"
    )

2020 -> 2022: +659 nodes
2022 -> 2024: +2,000 nodes
2024 -> 2026: +1,634 nodes


## Efficient overlap matrices

Instead of doing every pairwise set intersection, count overlaps by walking each common node once.

In [11]:
def build_overlap_data(
    old_hierarchy,
    new_hierarchy,
    level
):
    old_records = (
        records_by_id(
            old_hierarchy,
            level
        )
    )

    new_records = (
        records_by_id(
            new_hierarchy,
            level
        )
    )


    old_ids = sorted(
        old_records
    )

    new_ids = sorted(
        new_records
    )


    old_index = {
        cid: i
        for i, cid
        in enumerate(old_ids)
    }

    new_index = {
        cid: i
        for i, cid
        in enumerate(new_ids)
    }


    old_assignment = (
        node_assignment(
            old_hierarchy,
            level
        )
    )

    new_assignment = (
        node_assignment(
            new_hierarchy,
            level
        )
    )


    old_nodes = set(
        old_assignment
    )

    new_nodes = set(
        new_assignment
    )


    if not (
        old_nodes
        <=
        new_nodes
    ):
        raise ValueError(
            "Temporal node universe is not monotonic."
        )


    common_nodes = old_nodes


    intersections = np.zeros(
        (
            len(old_ids),
            len(new_ids)
        ),
        dtype=np.int64
    )


    for node_id in common_nodes:

        old_cluster = (
            old_assignment[
                node_id
            ]
        )

        new_cluster = (
            new_assignment[
                node_id
            ]
        )


        intersections[
            old_index[
                old_cluster
            ],
            new_index[
                new_cluster
            ]
        ] += 1


    old_sizes = (
        intersections.sum(
            axis=1
        )
    )

    new_common_sizes = (
        intersections.sum(
            axis=0
        )
    )


    union = (
        old_sizes[:, None]
        +
        new_common_sizes[None, :]
        -
        intersections
    )


    jaccard = np.divide(
        intersections,
        union,
        out=np.zeros_like(
            union,
            dtype=float
        ),
        where=union > 0
    )


    old_fraction = np.divide(
        intersections,
        old_sizes[:, None],
        out=np.zeros_like(
            intersections,
            dtype=float
        ),
        where=old_sizes[:, None] > 0
    )


    new_fraction = np.divide(
        intersections,
        new_common_sizes[None, :],
        out=np.zeros_like(
            intersections,
            dtype=float
        ),
        where=new_common_sizes[
            None,
            :
        ] > 0
    )


    return {
        "old_ids":
            old_ids,

        "new_ids":
            new_ids,

        "old_records":
            old_records,

        "new_records":
            new_records,

        "intersections":
            intersections,

        "old_sizes":
            old_sizes,

        "new_common_sizes":
            new_common_sizes,

        "jaccard":
            jaccard,

        "old_fraction":
            old_fraction,

        "new_fraction":
            new_fraction,

        "common_nodes":
            common_nodes,
    }

## Persistent ID generator

In [12]:
persistent_counters = {
    level: 0
    for level in LEVELS
}

In [13]:
def new_persistent_id(
    level
):
    index = (
        persistent_counters[
            level
        ]
    )

    persistent_counters[
        level
    ] += 1


    return (
        f"L{level}_C{index:05d}"
    )

## Initialize identities at 2020

In [14]:
persistent_maps = {
    2020: {}
}


for level in LEVELS:

    local_ids = sorted(
        records_by_id(
            static_hierarchies[
                2020
            ],
            level
        )
    )


    persistent_maps[
        2020
    ][level] = {
        local_id:
            new_persistent_id(
                level
            )

        for local_id
        in local_ids
    }

In [15]:
for level in LEVELS:

    mapping = (
        persistent_maps[
            2020
        ][level]
    )

    print(
        f"P{level}:",
        len(mapping),
        "initial persistent identities"
    )

P0: 12 initial persistent identities
P1: 60 initial persistent identities
P2: 300 initial persistent identities


## Parent consistency matrix

In [16]:
def parent_consistency_matrix(
    old_data,
    new_data,
    old_parent_map,
    new_parent_map
):
    matrix = np.zeros(
        (
            len(
                old_data[
                    "old_ids"
                ]
            ),
            len(
                old_data[
                    "new_ids"
                ]
            )
        ),
        dtype=float
    )


    for i, old_id in enumerate(
        old_data["old_ids"]
    ):

        old_parent_local = (
            old_data[
                "old_records"
            ][old_id][
                "parent_id"
            ]
        )


        old_parent_pid = (
            old_parent_map[
                old_parent_local
            ]
        )


        for j, new_id in enumerate(
            old_data["new_ids"]
        ):

            new_parent_local = (
                new_data[
                    new_id
                ][
                    "parent_id"
                ]
            )


            new_parent_pid = (
                new_parent_map[
                    new_parent_local
                ]
            )


            if (
                old_parent_pid
                ==
                new_parent_pid
            ):
                matrix[i, j] = 1.0


    return matrix

## Match one level

In [17]:
def match_level(
    old_hierarchy,
    new_hierarchy,
    level,
    old_parent_persistent_map=None,
    new_parent_persistent_map=None
):
    overlap = build_overlap_data(
        old_hierarchy,
        new_hierarchy,
        level
    )


    raw_similarity = (
        overlap[
            "jaccard"
        ]
    )


    if level == 0:

        score = (
            raw_similarity.copy()
        )


    else:

        parent_matrix = (
            parent_consistency_matrix(
                overlap,
                overlap[
                    "new_records"
                ],
                old_parent_persistent_map,
                new_parent_persistent_map
            )
        )


        score = (
            (1.0 - PARENT_BONUS)
            *
            raw_similarity

            +

            PARENT_BONUS
            *
            parent_matrix
        )


    row_ind, col_ind = (
        linear_sum_assignment(
            -score
        )
    )


    accepted_matches = []


    for i, j in zip(
        row_ind,
        col_ind
    ):

        raw_jaccard = float(
            raw_similarity[
                i,
                j
            ]
        )


        if (
            raw_jaccard
            >=
            MATCH_THRESHOLD
        ):

            accepted_matches.append({

                "old_id":
                    overlap[
                        "old_ids"
                    ][i],

                "new_id":
                    overlap[
                        "new_ids"
                    ][j],

                "jaccard":
                    raw_jaccard,

                "matching_score":
                    float(
                        score[
                            i,
                            j
                        ]
                    ),

                "intersection":
                    int(
                        overlap[
                            "intersections"
                        ][i, j]
                    ),
            })


    return (
        accepted_matches,
        overlap
    )

## Assign persistent identities

In [18]:
def inherit_persistent_ids(
    level,
    old_persistent_map,
    new_local_ids,
    matches
):
    new_map = {}


    for match in matches:

        old_id = (
            match[
                "old_id"
            ]
        )

        new_id = (
            match[
                "new_id"
            ]
        )


        new_map[
            new_id
        ] = (
            old_persistent_map[
                old_id
            ]
        )


    for new_id in sorted(
        new_local_ids
    ):

        if new_id not in new_map:

            new_map[
                new_id
            ] = (
                new_persistent_id(
                    level
                )
            )


    assert (
        len(
            set(
                new_map.values()
            )
        )
        ==
        len(new_map)
    )


    return new_map

## Detect significant overlap links

These links are used for split/merge detection, independently of the one-to-one Hungarian continuation match.

In [19]:
def significant_overlap_links(
    overlap
):
    old_links = defaultdict(list)
    new_links = defaultdict(list)


    intersections = (
        overlap[
            "intersections"
        ]
    )


    for i, old_id in enumerate(
        overlap[
            "old_ids"
        ]
    ):

        for j, new_id in enumerate(
            overlap[
                "new_ids"
            ]
        ):

            intersection = int(
                intersections[i, j]
            )


            if intersection == 0:
                continue


            old_share = float(
                overlap[
                    "old_fraction"
                ][i, j]
            )

            new_share = float(
                overlap[
                    "new_fraction"
                ][i, j]
            )


            if (
                old_share
                >=
                EVENT_OVERLAP_THRESHOLD

                or

                new_share
                >=
                EVENT_OVERLAP_THRESHOLD
            ):

                link = {
                    "old_id":
                        old_id,

                    "new_id":
                        new_id,

                    "intersection":
                        intersection,

                    "old_share":
                        old_share,

                    "new_share":
                        new_share,

                    "jaccard":
                        float(
                            overlap[
                                "jaccard"
                            ][i, j]
                        )
                }


                old_links[
                    old_id
                ].append(
                    link
                )

                new_links[
                    new_id
                ].append(
                    link
                )


    return (
        old_links,
        new_links
    )

## Explain event semantics

## Temporal event semantics

The event categories are not mutually exclusive.

A **persistent identity event** and a **structural event** can occur at the same
transition.

### Birth

A cluster at $t+1$ receives a new persistent identity because no previous cluster
was matched strongly enough.

### Death

A previous persistent identity is not inherited by any cluster at $t+1$.

### Growth

A persistent cluster survives and its total membership increases.

Growth is separated into:

- newly introduced corpus nodes;
- old nodes reassigned from other clusters.

### Split

One old cluster has significant overlap with two or more new clusters.

### Merge

One new cluster has significant overlap with two or more old clusters.

For a split or merge, one branch may retain the original persistent identity while
the remaining branches receive new identities. This makes identity continuation
deterministic while preserving the structural event in the event log.

## Event generation

In [21]:
def generate_events(
    old_year,
    new_year,
    level,
    old_hierarchy,
    new_hierarchy,
    old_persistent_map,
    new_persistent_map,
    matches,
    overlap
):
    events = []


    old_records = (
        overlap[
            "old_records"
        ]
    )

    new_records = (
        overlap[
            "new_records"
        ]
    )


    old_links, new_links = (
        significant_overlap_links(
            overlap
        )
    )


    matched_old = {
        m["old_id"]
        for m in matches
    }

    matched_new = {
        m["new_id"]
        for m in matches
    }


    # -------------------------
    # Birth
    # -------------------------

    for new_id in (
        overlap[
            "new_ids"
        ]
    ):

        if new_id not in matched_new:

            events.append({

                "from_year":
                    old_year,

                "to_year":
                    new_year,

                "level":
                    level,

                "event_type":
                    "birth",

                "source_persistent_ids":
                    [],

                "target_persistent_ids":
                    [
                        new_persistent_map[
                            new_id
                        ]
                    ],

                "source_local_ids":
                    [],

                "target_local_ids":
                    [
                        new_id
                    ],

                "new_size":
                    len(
                        new_records[
                            new_id
                        ][
                            "member_ids"
                        ]
                    ),

                "significant_predecessors":
                    len(
                        new_links.get(
                            new_id,
                            []
                        )
                    ),
            })


    # -------------------------
    # Death
    # -------------------------

    for old_id in (
        overlap[
            "old_ids"
        ]
    ):

        if old_id not in matched_old:

            events.append({

                "from_year":
                    old_year,

                "to_year":
                    new_year,

                "level":
                    level,

                "event_type":
                    "death",

                "source_persistent_ids":
                    [
                        old_persistent_map[
                            old_id
                        ]
                    ],

                "target_persistent_ids":
                    [],

                "source_local_ids":
                    [
                        old_id
                    ],

                "target_local_ids":
                    [],

                "old_size":
                    len(
                        old_records[
                            old_id
                        ][
                            "member_ids"
                        ]
                    ),

                "significant_successors":
                    len(
                        old_links.get(
                            old_id,
                            []
                        )
                    ),
            })


    # -------------------------
    # Split
    # -------------------------

    for old_id, links in (
        old_links.items()
    ):

        if len(links) < 2:
            continue


        target_ids = sorted(
            {
                link["new_id"]
                for link in links
            }
        )


        events.append({

            "from_year":
                old_year,

            "to_year":
                new_year,

            "level":
                level,

            "event_type":
                "split",

            "source_persistent_ids":
                [
                    old_persistent_map[
                        old_id
                    ]
                ],

            "target_persistent_ids":
                [
                    new_persistent_map[
                        new_id
                    ]
                    for new_id
                    in target_ids
                ],

            "source_local_ids":
                [
                    old_id
                ],

            "target_local_ids":
                target_ids,

            "overlaps":
                links,
        })


    # -------------------------
    # Merge
    # -------------------------

    for new_id, links in (
        new_links.items()
    ):

        if len(links) < 2:
            continue


        source_ids = sorted(
            {
                link["old_id"]
                for link in links
            }
        )


        events.append({

            "from_year":
                old_year,

            "to_year":
                new_year,

            "level":
                level,

            "event_type":
                "merge",

            "source_persistent_ids":
                [
                    old_persistent_map[
                        old_id
                    ]
                    for old_id
                    in source_ids
                ],

            "target_persistent_ids":
                [
                    new_persistent_map[
                        new_id
                    ]
                ],

            "source_local_ids":
                source_ids,

            "target_local_ids":
                [
                    new_id
                ],

            "overlaps":
                links,
        })


    # -------------------------
    # Growth
    # -------------------------

    old_node_universe = (
        overlap[
            "common_nodes"
        ]
    )


    for match in matches:

        old_id = (
            match[
                "old_id"
            ]
        )

        new_id = (
            match[
                "new_id"
            ]
        )


        old_members = set(
            old_records[
                old_id
            ][
                "member_ids"
            ]
        )

        new_members = set(
            new_records[
                new_id
            ][
                "member_ids"
            ]
        )


        if (
            len(new_members)
            <=
            len(old_members)
        ):
            continue


        introduced_nodes = (
            new_members
            -
            old_node_universe
        )


        reassigned_in = (
            (
                new_members
                &
                old_node_universe
            )
            -
            old_members
        )


        reassigned_out = (
            old_members
            -
            new_members
        )


        events.append({

            "from_year":
                old_year,

            "to_year":
                new_year,

            "level":
                level,

            "event_type":
                "growth",

            "source_persistent_ids":
                [
                    old_persistent_map[
                        old_id
                    ]
                ],

            "target_persistent_ids":
                [
                    new_persistent_map[
                        new_id
                    ]
                ],

            "source_local_ids":
                [
                    old_id
                ],

            "target_local_ids":
                [
                    new_id
                ],

            "old_size":
                len(
                    old_members
                ),

            "new_size":
                len(
                    new_members
                ),

            "retained_nodes":
                len(
                    old_members
                    &
                    new_members
                ),

            "introduced_nodes":
                len(
                    introduced_nodes
                ),

            "reassigned_in_nodes":
                len(
                    reassigned_in
                ),

            "reassigned_out_nodes":
                len(
                    reassigned_out
                ),

            "jaccard":
                match[
                    "jaccard"
                ],
        })


    return events

## Transition stability diagnostics

In [22]:
def transition_diagnostics(
    old_hierarchy,
    new_hierarchy,
    level,
    old_persistent_map,
    new_persistent_map,
    matches,
    overlap
):
    old_assignment = (
        node_assignment(
            old_hierarchy,
            level
        )
    )

    new_assignment = (
        node_assignment(
            new_hierarchy,
            level
        )
    )


    common_nodes = (
        overlap[
            "common_nodes"
        ]
    )


    same_identity_nodes = 0


    for node_id in common_nodes:

        old_local = (
            old_assignment[
                node_id
            ]
        )

        new_local = (
            new_assignment[
                node_id
            ]
        )


        if (
            old_persistent_map[
                old_local
            ]
            ==
            new_persistent_map[
                new_local
            ]
        ):

            same_identity_nodes += 1


    node_identity_retention = (
        same_identity_nodes
        /
        len(common_nodes)
        if common_nodes
        else 1.0
    )


    jaccards = [
        match["jaccard"]
        for match in matches
    ]


    return {

        "old_clusters":
            len(
                overlap[
                    "old_ids"
                ]
            ),

        "new_clusters":
            len(
                overlap[
                    "new_ids"
                ]
            ),

        "matched_clusters":
            len(matches),

        "cluster_identity_retention":
            (
                len(matches)
                /
                len(
                    overlap[
                        "old_ids"
                    ]
                )
            ),

        "mean_match_jaccard":
            float(
                np.mean(jaccards)
            )
            if jaccards
            else 0.0,

        "median_match_jaccard":
            float(
                np.median(jaccards)
            )
            if jaccards
            else 0.0,

        "node_identity_retention":
            float(
                node_identity_retention
            ),
    }

These are useful diagnostics, but T6 will still perform the formal cross-snapshot stability evaluation, including ARI and confidence intervals, as required by the assessment.

## Couple all snapshots sequentially

In [23]:
temporal_events = []

temporal_diagnostics = {}

match_records = {}

In [24]:
for old_year, new_year in zip(
    SNAPSHOT_YEARS[:-1],
    SNAPSHOT_YEARS[1:]
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"TEMPORAL COUPLING "
        f"{old_year} -> {new_year}"
    )

    print(
        "=" * 70
    )


    start_time = (
        time.perf_counter()
    )


    persistent_maps[
        new_year
    ] = {}


    temporal_diagnostics[
        f"{old_year}->{new_year}"
    ] = {}


    match_records[
        f"{old_year}->{new_year}"
    ] = {}


    for level in LEVELS:

        if level == 0:

            old_parent_map = None
            new_parent_map = None

        else:

            old_parent_map = (
                persistent_maps[
                    old_year
                ][
                    level - 1
                ]
            )

            new_parent_map = (
                persistent_maps[
                    new_year
                ][
                    level - 1
                ]
            )


        matches, overlap = (
            match_level(
                static_hierarchies[
                    old_year
                ],
                static_hierarchies[
                    new_year
                ],
                level,
                old_parent_map,
                new_parent_map
            )
        )


        new_local_ids = (
            records_by_id(
                static_hierarchies[
                    new_year
                ],
                level
            ).keys()
        )


        new_persistent_map = (
            inherit_persistent_ids(
                level,
                persistent_maps[
                    old_year
                ][level],
                new_local_ids,
                matches
            )
        )


        persistent_maps[
            new_year
        ][level] = (
            new_persistent_map
        )


        events = (
            generate_events(
                old_year,
                new_year,
                level,
                static_hierarchies[
                    old_year
                ],
                static_hierarchies[
                    new_year
                ],
                persistent_maps[
                    old_year
                ][level],
                new_persistent_map,
                matches,
                overlap
            )
        )


        temporal_events.extend(
            events
        )


        diagnostics = (
            transition_diagnostics(
                static_hierarchies[
                    old_year
                ],
                static_hierarchies[
                    new_year
                ],
                level,
                persistent_maps[
                    old_year
                ][level],
                new_persistent_map,
                matches,
                overlap
            )
        )


        temporal_diagnostics[
            f"{old_year}->{new_year}"
        ][
            f"P{level}"
        ] = diagnostics


        match_records[
            f"{old_year}->{new_year}"
        ][
            f"P{level}"
        ] = matches


        print(
            f"P{level}: "
            f"{len(matches)} identities matched | "
            f"node retention="
            f"{diagnostics['node_identity_retention']:.3f}"
        )


    temporal_diagnostics[
        f"{old_year}->{new_year}"
    ][
        "runtime_seconds"
    ] = float(
        time.perf_counter()
        -
        start_time
    )


TEMPORAL COUPLING 2020 -> 2022
P0: 11 identities matched | node retention=0.717
P1: 48 identities matched | node retention=0.794
P2: 215 identities matched | node retention=0.848

TEMPORAL COUPLING 2022 -> 2024
P0: 7 identities matched | node retention=0.653
P1: 27 identities matched | node retention=0.428
P2: 173 identities matched | node retention=0.658

TEMPORAL COUPLING 2024 -> 2026
P0: 10 identities matched | node retention=0.915
P1: 43 identities matched | node retention=0.766
P2: 238 identities matched | node retention=0.778


## Initial birth events

For lifecycle completeness, record the first appearance of every 2020 supernode.

In [25]:
initial_events = []


for level in LEVELS:

    records = (
        records_by_id(
            static_hierarchies[
                2020
            ],
            level
        )
    )


    for local_id in sorted(
        records
    ):

        initial_events.append({

            "from_year":
                None,

            "to_year":
                2020,

            "level":
                level,

            "event_type":
                "birth",

            "source_persistent_ids":
                [],

            "target_persistent_ids":
                [
                    persistent_maps[
                        2020
                    ][level][
                        local_id
                    ]
                ],

            "source_local_ids":
                [],

            "target_local_ids":
                [
                    local_id
                ],

            "initial_snapshot":
                True,
        })

Put them first:

In [26]:
temporal_events = (
    initial_events
    +
    temporal_events
)

## Event summary

In [27]:
event_summary = Counter(
    event[
        "event_type"
    ]
    for event
    in temporal_events
)


event_summary

Counter({'birth': 716,
         'death': 344,
         'split': 277,
         'merge': 310,
         'growth': 434})

In [28]:
event_summary_df = (
    pd.DataFrame(
        [
            {
                "event_type":
                    event_type,

                "count":
                    count
            }

            for event_type, count
            in sorted(
                event_summary.items()
            )
        ]
    )
)


event_summary_df

,event_type,count
0,birth,716
1,death,344
2,growth,434
3,merge,310
4,split,277


## Diagnostic table

In [29]:
diagnostic_rows = []


for transition, levels in (
    temporal_diagnostics.items()
):

    for level in [
        "P0",
        "P1",
        "P2"
    ]:

        row = {
            "transition":
                transition,

            "level":
                level,

            **levels[level]
        }

        diagnostic_rows.append(
            row
        )


temporal_diagnostics_df = (
    pd.DataFrame(
        diagnostic_rows
    )
)


temporal_diagnostics_df

,transition,level,old_clusters,new_clusters,matched_clusters,cluster_identity_retention,mean_match_jaccard,median_match_jaccard,node_identity_retention
0,2020->2022,P0,12,12,11,0.916667,0.593500,0.592593,0.716944
1,2020->2022,P1,60,60,48,0.800000,0.742585,0.839080,0.794020
2,2020->2022,P2,300,300,215,0.716667,0.878432,1.000000,0.847841
3,2022->2024,P0,12,12,7,0.583333,0.510842,0.435714,0.652957
4,2022->2024,P1,60,60,27,0.450000,0.649623,0.764706,0.428373
5,2022->2024,P2,300,300,173,0.576667,0.771912,0.898876,0.658041
6,2024->2026,P0,12,12,10,0.833333,0.673795,0.688863,0.914505
7,2024->2026,P1,60,60,43,0.716667,0.680654,0.714286,0.765610
8,2024->2026,P2,300,300,238,0.793333,0.738385,0.795833,0.777618


## Validate persistent identity uniqueness

In [30]:
for year in SNAPSHOT_YEARS:

    for level in LEVELS:

        mapping = (
            persistent_maps[
                year
            ][level]
        )


        persistent_ids = list(
            mapping.values()
        )


        assert (
            len(
                persistent_ids
            )
            ==
            len(
                set(
                    persistent_ids
                )
            )
        )


        print(
            year,
            f"P{level}:",
            "persistent IDs unique"
        )

2020 P0: persistent IDs unique
2020 P1: persistent IDs unique
2020 P2: persistent IDs unique
2022 P0: persistent IDs unique
2022 P1: persistent IDs unique
2022 P2: persistent IDs unique
2024 P0: persistent IDs unique
2024 P1: persistent IDs unique
2024 P2: persistent IDs unique
2026 P0: persistent IDs unique
2026 P1: persistent IDs unique
2026 P2: persistent IDs unique


## Validate identity continuity

In [31]:
for old_year, new_year in zip(
    SNAPSHOT_YEARS[:-1],
    SNAPSHOT_YEARS[1:]
):

    transition = (
        f"{old_year}->{new_year}"
    )


    for level in LEVELS:

        for match in (
            match_records[
                transition
            ][
                f"P{level}"
            ]
        ):

            old_pid = (
                persistent_maps[
                    old_year
                ][level][
                    match[
                        "old_id"
                    ]
                ]
            )


            new_pid = (
                persistent_maps[
                    new_year
                ][level][
                    match[
                        "new_id"
                    ]
                ]
            )


            assert (
                old_pid
                ==
                new_pid
            )


print(
    "Persistent identity inheritance "
    "validation passed."
)

Persistent identity inheritance validation passed.


## Serialize temporally coupled hierarchy

In [32]:
def serialize_temporal_hierarchy(
    year
):
    source = (
        static_hierarchies[
            year
        ]
    )


    output = {
        "year":
            year,

        "static":
            False,

        "temporally_coupled":
            True,

        "temporal_method":
            (
                "overlap-based Hungarian matching "
                "with top-down parent-consistency bonus"
            ),

        "levels": {}
    }


    # -------------------------
    # P0, P1, P2
    # -------------------------

    for level in LEVELS:

        output[
            "levels"
        ][
            str(level)
        ] = []


        for record in (
            source[
                "levels"
            ][
                str(level)
            ]
        ):

            local_id = (
                record["id"]
            )


            persistent_id = (
                persistent_maps[
                    year
                ][level][
                    local_id
                ]
            )


            if level == 0:

                parent_persistent_id = (
                    None
                )

            else:

                parent_local_id = (
                    record[
                        "parent_id"
                    ]
                )


                parent_persistent_id = (
                    persistent_maps[
                        year
                    ][
                        level - 1
                    ][
                        parent_local_id
                    ]
                )


            new_record = dict(
                record
            )


            new_record[
                "persistent_id"
            ] = (
                persistent_id
            )


            new_record[
                "parent_persistent_id"
            ] = (
                parent_persistent_id
            )


            output[
                "levels"
            ][
                str(level)
            ].append(
                new_record
            )


    # -------------------------
    # Finest node level
    # -------------------------

    output[
        "levels"
    ][
        "3"
    ] = []


    for record in (
        source[
            "levels"
        ][
            "3"
        ]
    ):

        new_record = dict(
            record
        )


        parent_local_id = (
            record[
                "parent_id"
            ]
        )


        new_record[
            "persistent_id"
        ] = (
            record["id"]
        )


        new_record[
            "parent_persistent_id"
        ] = (
            persistent_maps[
                year
            ][2][
                parent_local_id
            ]
        )


        output[
            "levels"
        ][
            "3"
        ].append(
            new_record
        )


    return output

## Validate persistent parent relationships

In [33]:
def validate_temporal_hierarchy(
    hierarchy
):
    for level in [
        0,
        1,
        2
    ]:

        records = (
            hierarchy[
                "levels"
            ][
                str(level)
            ]
        )


        persistent_ids = {
            record[
                "persistent_id"
            ]
            for record
            in records
        }


        assert (
            len(
                persistent_ids
            )
            ==
            len(records)
        )


        if level > 0:

            parent_ids = {
                record[
                    "persistent_id"
                ]
                for record
                in hierarchy[
                    "levels"
                ][
                    str(
                        level - 1
                    )
                ]
            }


            for record in records:

                assert (
                    record[
                        "parent_persistent_id"
                    ]
                    in
                    parent_ids
                )


    print(
        hierarchy["year"],
        "temporal hierarchy valid"
    )

## Save hierarchy per snapshot

In [34]:
temporal_hierarchies = {}


for year in SNAPSHOT_YEARS:

    hierarchy = (
        serialize_temporal_hierarchy(
            year
        )
    )


    validate_temporal_hierarchy(
        hierarchy
    )


    temporal_hierarchies[
        year
    ] = hierarchy


    path = (
        TEMPORAL_DIR
        /
        f"hierarchy_{year}_temporal_unlabelled.json"
    )


    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            hierarchy,
            f,
            indent=2
        )


    print(
        "Saved:",
        path
    )

2020 temporal hierarchy valid
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/hierarchy_2020_temporal_unlabelled.json
2022 temporal hierarchy valid
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/hierarchy_2022_temporal_unlabelled.json
2024 temporal hierarchy valid
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/hierarchy_2024_temporal_unlabelled.json
2026 temporal hierarchy valid
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/hierarchy_2026_temporal_unlabelled.json


## Save temporal event log

In [35]:
EVENT_PATH = (
    TEMPORAL_DIR
    /
    "temporal_events.json"
)


with open(
    EVENT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        temporal_events,
        f,
        indent=2
    )


print(
    "Saved:",
    EVENT_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/temporal_events.json


## Save diagnostics

In [36]:
DIAGNOSTICS_PATH = (
    TEMPORAL_DIR
    /
    "temporal_coupling_diagnostics.json"
)


with open(
    DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        temporal_diagnostics,
        f,
        indent=2
    )


print(
    "Saved:",
    DIAGNOSTICS_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/temporal_coupling_diagnostics.json


## Save coupling configuration

In [37]:
temporal_config = {

    "snapshot_years":
        SNAPSHOT_YEARS,

    "levels_coupled":
        LEVELS,

    "matching_method":
        "Hungarian maximum-weight matching",

    "membership_similarity":
        "Jaccard over previously visible nodes only",

    "match_threshold":
        MATCH_THRESHOLD,

    "event_overlap_threshold":
        EVENT_OVERLAP_THRESHOLD,

    "parent_bonus":
        PARENT_BONUS,

    "matching_order":
        "coarse-to-fine",

    "gold_benchmark_used":
        False,

    "partition_memberships_modified":
        False,
}

In [38]:
CONFIG_PATH = (
    TEMPORAL_DIR
    /
    "temporal_coupling_config.json"
)


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        temporal_config,
        f,
        indent=2
    )


print(
    "Saved:",
    CONFIG_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/temporal/temporal_coupling_config.json


## Complexity

## Computational cost

Let $K_\ell$ denote the number of supernodes at level $\ell$.

For each transition and hierarchy level:

1. overlap counts are constructed by scanning the common nodes once;

$$
O(|V_t|);
$$

2. the matching matrix requires

$$
O(K_\ell^2)
$$

memory;

3. Hungarian assignment requires

$$
O(K_\ell^3)
$$

time in the worst case.

The largest matched level contains only

$$
K_2=300
$$

supernodes, so this cost is small for the supplied corpus.

No semantic embeddings, gold benchmark answers, or future snapshots beyond
$t+1$ are used to establish identity at transition $t\rightarrow t+1$.

## Guarantees and limitations

## What temporal coupling guarantees

### Guaranteed by construction

- each persistent identity appears at most once per hierarchy level and snapshot;
- accepted continuation matches inherit exactly one previous persistent ID;
- matching proceeds sequentially, so snapshot $t+1$ never uses information from
  later snapshots;
- finer-level identity matching is biased toward already-matched persistent
  parents;
- birth, death, growth, split, and merge events are explicitly logged.

### Empirical rather than guaranteed

The matching layer does **not** change the cluster memberships produced by the
static optimizer in Section 06.

Therefore it guarantees **trackable identity continuity**, but it does not
mathematically guarantee that independently produced partitions are similar.

Whether the hierarchy genuinely evolves smoothly is an empirical question and
will be evaluated in T6 using real cross-snapshot stability metrics.

If stability proves insufficient, the natural stronger alternative is to add a
temporal regularisation term or warm-start the coarsening optimization itself.

## Final summary

In [39]:
print(
    "===== TEMPORAL COUPLING SUMMARY ====="
)

print(
    "Snapshots:",
    SNAPSHOT_YEARS
)

print(
    "Matched levels:",
    LEVELS
)

print(
    "Match threshold:",
    MATCH_THRESHOLD
)

print(
    "Parent bonus:",
    PARENT_BONUS
)

print(
    "Total temporal events:",
    len(
        temporal_events
    )
)

print(
    "Event types:",
    dict(
        event_summary
    )
)

print(
    "\nSection 07 temporal coupling passed."
)

===== TEMPORAL COUPLING SUMMARY =====
Snapshots: [2020, 2022, 2024, 2026]
Matched levels: [0, 1, 2]
Match threshold: 0.2
Parent bonus: 0.15
Total temporal events: 2081
Event types: {'birth': 716, 'death': 344, 'split': 277, 'merge': 310, 'growth': 434}

Section 07 temporal coupling passed.


## What we now have

Before temporal coupling, the four snapshot hierarchies were independent:

```text
2020 hierarchy     2022 hierarchy     2024 hierarchy     2026 hierarchy
      ?                   ?                  ?                  ?

to:

L0_C00003 ─────────▶ L0_C00003 ─────────▶ L0_C00003 ─────────▶ L0_C00003
                         │
                         ├── growth
                         │
                         └── split ───────▶ L0_C00012

L0_C00007 ───────────── merge ───────────▶ L0_C00005


```
So supernodes now have persistent identities and explicit evolutionary histories, rather than being unrelated cluster numbers at each snapshot.

This makes the temporal dimension real, which the assessment explicitly identifies as a major part of the PhD-level task and assigns substantial weight to.

## Git Push

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/03_t1_descriptive_statistics.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/03_t1_descriptive_statistics.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())